# Transpile and Compare

## Context and motivation

A quantum circuit written using gates such as `H` and `CX` is an **abstract circuit**. Real quantum hardware cannot necessarily implement these gates directly.

Instead, each quantum computer has a set of **native gates** that it can perform efficiently. The hardware also has a specific physical layout, meaning that some pairs of qubits can interact directly while others cannot.

In Qiskit, the `transpile()` function converts your circuit into a form that is compatible with the hardware constraints you specify.

For example:

```python
transpiled = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cx']
)
```

Here, `transpile()` takes the original circuit `qc` and rewrites it using the specified basis gates.

You can also give it a **coupling map** to tell it which qubits are physically connected:

```python
transpiled = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cx'],
    coupling_map=coupling_map
)
```

The transpiler may therefore need to:

1. **Decompose gates:** replace gates with combinations of gates supported by the hardware.
2. **Route interactions:** add operations such as `SWAP` gates when two qubits need to interact but are not directly connected.
3. **Optimise the circuit:** remove unnecessary operations or find a more efficient implementation where possible.

This exercise will use `transpile()` to see how these hardware constraints change the same quantum circuit.

## Problem statement

You will first reuse your GHZ-state circuit from Q9 in the `Qiskit_Intro` notebook from Day 1.

You will then use Qiskit's `transpile()` function to compile the circuit under different hardware constraints and compare the results.

### Part 1: Start with the original circuit

Build your GHZ-state circuit from Q9 and print:

```python
qc.depth()
qc.size()
```

These give:

* `depth()`: the number of sequential layers of gates
* `size()`: the total number of gates

Keep these values so you can compare them with the transpiled circuits.

---

### Part 2: Different native gates

First, transpile the **same original GHZ circuit** using:

```python
transpiled = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cx']
)
```

This is representative of a gate set used by superconducting-qubit hardware.

Print:

```python
transpiled.depth()
transpiled.size()
transpiled.count_ops()
```

Compare these with the original circuit.

Now transpile the **same original circuit again**, but using:

```python
transpiled = transpile(
    qc,
    basis_gates=['rx', 'ry', 'rz', 'rxx']
)
```

This uses a different set of single- and two-qubit gates, illustrating how a different hardware platform can require a different decomposition of the same abstract circuit.

Again, print the depth, size, and gate counts.

### Questions

* Did the depth change?
* Did the total number of gates change?
* Which gates were used in each version?
* Why might the same quantum operation require a different sequence of physical gates on different hardware?

---

### Part 3: Limited qubit connectivity

Now we will look at a different source of overhead: **connectivity**.

Imagine three physical qubits arranged in a line:

```text
0 ─── 1 ─── 2
```

Qubit 0 can interact directly with qubit 1, and qubit 1 can interact directly with qubit 2.

However, **qubit 0 and qubit 2 cannot interact directly**.

Build a 3-qubit circuit containing interactions between all three pairs:

```python
cx(0, 1)
cx(1, 2)
cx(0, 2)
```

The final `cx(0, 2)` is important: qubits 0 and 2 are not directly connected in the linear layout.

#### First: no connectivity constraint

Use `transpile()` without specifying a coupling map:

```python
transpiled = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cx']
)
```

Record:

```python
transpiled.depth()
transpiled.size()
transpiled.count_ops()
```

#### Second: impose the linear connectivity

Create the linear coupling map:

```python
coupling_map = CouplingMap.from_line(3)
```

Then use it with `transpile()`:

```python
transpiled = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cx'],
    coupling_map=coupling_map
)
```

Again, record the depth, size, and gate counts.

### Questions

Compare the unconstrained and constrained circuits.

1. Did the depth increase?
2. Did the total number of gates increase?
3. What additional gates appeared?
4. Why were those gates necessary?

### What should happen?

The constrained circuit needs to implement an interaction between qubits 0 and 2, even though those qubits are not directly connected.

The transpiler therefore needs to **move the quantum states between physical qubits**, typically using `SWAP` operations, so that the required interaction can take place.

For example, schematically:

```text
0 ─── 1 ─── 2

       ↓ SWAP

0 ─── 2 ─── 1
```

The exact transpiled circuit may use a different sequence, so inspect `count_ops()` and the circuit itself rather than assuming a particular implementation.

This is an example of **routing overhead**: the quantum algorithm has not changed, but the hardware constraints mean that additional operations are required to implement it.

---

## Part 4: How does this relate to real quantum hardware?

The two sources of overhead you have seen correspond to real differences between quantum-computing platforms.

### Example 1: IBM superconducting qubits

IBM's quantum processors use **superconducting qubits** arranged in a specific physical connectivity pattern. A qubit is generally only able to perform a two-qubit gate directly with certain neighbouring qubits.

For example:

```text
0 ─── 1 ─── 2
```

A two-qubit gate between qubits 0 and 1 is directly available, but a gate between qubits 0 and 2 is not.

If your algorithm requires:

```python
cx(0, 2)
```

the `transpile()` function has to find another way to implement it, typically by using **SWAP operations** to move the quantum states until the required qubits become neighbours.

This is why the physical layout of IBM hardware can affect the depth and gate count of your circuit.

### Example 2: Trapped-ion quantum computers

Trapped-ion systems use a very different physical architecture.

Individual ions are held together in a trap and can interact through their shared motion. Because of this, trapped-ion processors can often implement two-qubit interactions between **non-neighbouring ions**, rather than being restricted to a fixed nearest-neighbour connectivity graph in the same way as many superconducting processors.

The trade-off is that they use a different set of native operations. For example, a trapped-ion system may naturally implement operations involving rotations and an XX-type interaction such as:

```text
RXX(θ)
```

rather than using `CX` as its fundamental two-qubit operation.

So the same abstract circuit can look quite different after transpilation:

**Superconducting hardware:**

```text
H, CX, CX
      ↓
transpile()
      ↓
native gates + possible SWAPs
```

**Trapped-ion hardware:**

```text
H, CX, CX
      ↓
transpile()
      ↓
single-qubit rotations + XX-type interactions
```

The important point is that **different hardware platforms have different physical constraints**, so `transpile()` converts the same high-level quantum algorithm into different low-level circuits.

---

## Key takeaway

The Qiskit `transpile()` function is the bridge between your **abstract quantum circuit** and the **constraints of real hardware**.

It can:

**1. Decompose gates**

```text
Abstract gates
      ↓
Hardware-compatible gates
```

**2. Route qubits**

```text
Required interaction
      ↓
Hardware connectivity
      ↓
Possible SWAP operations
```

**3. Optimise the resulting circuit**

The same quantum algorithm can therefore require different numbers of gates and different execution depths depending on the hardware it is run on.

In this exercise, you have deliberately changed these constraints yourself so that you can see the resulting overhead.


In [ ]:
!pip install qiskit --quiet
!pip install qiskit[visualization] --quiet


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap

# STUDENT TASK 1: build the GHZ circuit from Q10 and print depth/size
# YOUR CODE HERE


In [ ]:
# STUDENT TASK 2: transpile for a superconducting-style basis gate set
basis_ibm = ['rz', 'sx', 'x', 'cx']
# YOUR CODE HERE (use transpile(), print depth/size/count_ops())


In [ ]:
# STUDENT TASK 3: transpile the SAME original circuit for a trapped-ion-style basis
basis_ion = ['rx', 'ry', 'rz', 'rxx']
# YOUR CODE HERE


In [ ]:
# STUDENT TASK 4a: build a circuit needing all 3 pairs of qubits to interact directly
# (a "triangle" of entangling gates: (0,1), (1,2), (0,2))
# YOUR CODE HERE


In [ ]:
# STUDENT TASK 4b: transpile it with the superconducting basis, no coupling constraint
# YOUR CODE HERE


In [ ]:
# STUDENT TASK 4c: transpile it again, this time with coupling_map=CouplingMap.from_line(3)
# Compare depth/size/count_ops() to the unconstrained version above
# YOUR CODE HERE


## Discussion

- For the GHZ circuit (Task 2 vs 3): which basis gate set needed more total gates? Does that necessarily mean it's "worse" hardware β€” or does it depend on how fast/reliable each native gate is in practice?
- For the triangle circuit (Task 4): the linear-coupling version should need at least one extra two-qubit gate compared to the unconstrained version. That's the cost of **routing**: the transpiler had to insert extra gates (functionally acting like a SWAP) to get two non-adjacent qubits to interact. This is exactly why real-hardware qubit *layout and connectivity* matters as much as raw qubit count.
- **Bonus:** go back to the original GHZ circuit (star-shaped: one qubit controls both others) and transpile it with the *same* linear coupling map. You should find it needs **no extra routing at all**. Can you see why the GHZ circuit's structure is naturally compatible with a line topology, while the triangle circuit isn't?


In [ ]:
# Bonus: transpile the original GHZ circuit (not the triangle one) with the linear coupling map
# and compare to its unconstrained transpilation
# YOUR CODE HERE
